### The index-set search for $N(s)$ and $T(s)$.

Let $N=N(s)$ and $T=T(s)$ be two matrices with the same number $n$ of rows. This notebook searches for a row-index set $I\subseteq[n]$ and its complement
$$S=[n]\setminus I.$$
The rank gap associated with $I$ is defined as
$$\operatorname{gap}(I)=\left(\operatorname{rank}(N)-\operatorname{rank}(N[I,:])\right)-
\left(\operatorname{rank}(T)-\operatorname{rank}(T[I,:])\right).$$

The final cell performs the search and returns the resulting index sets $I$ and $S$ satisfying the prescribed gap constraint
$$a\leq \operatorname{gap}(I)\leq b.$$

In [ ]:
using Serialization
using Random
using DelimitedFiles

In [ ]:
# ============================================================
# User configuration
# ============================================================

# Change only these two paths when testing another pair of matrices.
# The common number of rows is detected automatically.
N_FILE = "Ns.jls"
T_FILE = "Ts.jls"

In [ ]:
# Preferred field names used when a serialized object is a
# NamedTuple or Dict instead of a matrix itself.
N_PREFERRED_KEYS = (
    :N_rat,
    :Ns_rat,
    :N,
    :Ns,
    :nullspace,
    :matrix,
)

T_PREFERRED_KEYS = (
    :T_rat,
    :Ts_rat,
    :T,
    :Ts,
    :tangent,
    :matrix,
)


In [ ]:
# ============================================================
# Load a matrix from a Julia Serialization file.
# ============================================================

function extract_matrix(
    obj;
    preferred_keys = (),
    object_name::AbstractString = "object",
)
    if obj isa AbstractMatrix
        return obj
    end

    if obj isa NamedTuple
        for key in preferred_keys
            if hasproperty(obj, key)
                value = getproperty(obj, key)
                if value isa AbstractMatrix
                    return value
                end
            end
        end

        candidates = [
            (key, getproperty(obj, key))
            for key in propertynames(obj)
            if getproperty(obj, key) isa AbstractMatrix
        ]

        if length(candidates) == 1
            return candidates[1][2]
        elseif isempty(candidates)
            error(
                "$object_name is a NamedTuple, but it contains no matrix. " *
                "Available fields: $(propertynames(obj))",
            )
        else
            candidate_names = first.(candidates)
            error(
                "$object_name contains several matrices: $candidate_names. " *
                "Add the desired field name to preferred_keys.",
            )
        end
    end

    if obj isa AbstractDict
        for key in preferred_keys
            for candidate_key in (key, String(key))
                if haskey(obj, candidate_key)
                    value = obj[candidate_key]
                    if value isa AbstractMatrix
                        return value
                    end
                end
            end
        end

        candidates = [
            (key, value)
            for (key, value) in pairs(obj)
            if value isa AbstractMatrix
        ]

        if length(candidates) == 1
            return candidates[1][2]
        elseif isempty(candidates)
            error(
                "$object_name is a dictionary, but it contains no matrix. " *
                "Available keys: $(collect(keys(obj)))",
            )
        else
            candidate_names = first.(candidates)
            error(
                "$object_name contains several matrices: $candidate_names. " *
                "Add the desired key to preferred_keys.",
            )
        end
    end

    if obj isa Tuple
        candidates = [
            value
            for value in obj
            if value isa AbstractMatrix
        ]

        if length(candidates) == 1
            return candidates[1]
        elseif isempty(candidates)
            error("$object_name is a tuple, but it contains no matrix.")
        else
            error(
                "$object_name is a tuple containing several matrices. " *
                "Store the desired matrix separately or use a NamedTuple.",
            )
        end
    end

    error(
        "Cannot extract a matrix from $object_name of type $(typeof(obj)).",
    )
end

In [ ]:
function load_serialized_matrix(
    filename::AbstractString;
    preferred_keys = (),
    object_name::AbstractString = filename,
)
    isfile(filename) || error("File not found: $filename")

    obj = deserialize(filename)

    return extract_matrix(
        obj;
        preferred_keys = preferred_keys,
        object_name = object_name,
    )
end

In [ ]:
N = load_serialized_matrix(
    N_FILE;
    preferred_keys = N_PREFERRED_KEYS,
    object_name = "N",
)

In [ ]:
T = load_serialized_matrix(
    T_FILE;
    preferred_keys = T_PREFERRED_KEYS,
    object_name = "T",
)

In [ ]:
@assert ndims(N) == 2
@assert ndims(T) == 2
@assert size(N, 1) == size(T, 1) (
    "N and T must have the same number of rows, but their sizes are " *
    "$(size(N)) and $(size(T))."
)
@assert size(N, 1) > 0 "The matrices must have at least one row."

println("size(N) = ", size(N), ", eltype(N) = ", eltype(N))
println("size(T) = ", size(T), ", eltype(T) = ", eltype(T))
println("Number of common row coordinates = ", size(N, 1))

In [ ]:
# ============================================================
# Convert exact or floating-point entries modulo a prime.
#
# Rational entries are mapped by
#     a/b  ->  a * inv(b) mod p.
#
# This avoids constructing a potentially enormous common
# denominator and works for arbitrary row counts.
# ============================================================

function to_big_rational(
    x;
    float_tolerance::Real = 1e-12,
)
    if x isa Rational
        return (
            BigInt(numerator(x)) //
            BigInt(denominator(x))
        )
    elseif x isa Integer
        return BigInt(x) // BigInt(1)
    elseif x isa AbstractFloat
        isfinite(x) || error("Nonfinite floating-point entry: $x")
        return rationalize(
            BigInt,
            x;
            tol = float_tolerance,
        )
    end

    # This fallback supports exact scalar types that implement
    # numerator and denominator, for example some CAS rationals.
    try
        return (
            BigInt(numerator(x)) //
            BigInt(denominator(x))
        )
    catch
        error(
            "Unsupported matrix entry type $(typeof(x)). " *
            "Supported types include integers, rationals, and floats.",
        )
    end
end

In [ ]:
function scalar_mod_prime(
    x,
    p::Int;
    float_tolerance::Real = 1e-12,
)
    q = to_big_rational(
        x;
        float_tolerance = float_tolerance,
    )

    numerator_mod = Int(mod(numerator(q), p))
    denominator_mod = Int(mod(denominator(q), p))

    denominator_mod != 0 || throw(
        DomainError(
            p,
            "The prime divides an entry denominator.",
        ),
    )

    return mod(
        numerator_mod * invmod(denominator_mod, p),
        p,
    )
end

In [ ]:
function matrix_mod_prime(
    M::AbstractMatrix,
    p::Int;
    float_tolerance::Real = 1e-12,
)
    A = Matrix{Int}(undef, size(M, 1), size(M, 2))

    for j in axes(M, 2), i in axes(M, 1)
        A[i, j] = scalar_mod_prime(
            M[i, j],
            p;
            float_tolerance = float_tolerance,
        )
    end

    return A
end

In [ ]:
# ============================================================
# Modular row-rank tools.
# ============================================================

mutable struct ModBasis
    p::Int
    ncols::Int
    basis::Vector{Union{Nothing, Vector{Int}}}
    rank::Int
end


function ModBasis(p::Int, ncols::Int)
    return ModBasis(
        p,
        ncols,
        Union{Nothing, Vector{Int}}[
            nothing for _ in 1:ncols
        ],
        0,
    )
end

In [ ]:
function reduce_with_basis!(
    v::Vector{Int},
    state::ModBasis,
)
    p = state.p
    ncols = state.ncols

    @inbounds for j in 1:ncols
        if v[j] != 0 && state.basis[j] !== nothing
            basis_row = state.basis[j]::Vector{Int}
            coefficient = v[j]

            for k in j:ncols
                v[k] = mod(
                    v[k] - coefficient * basis_row[k],
                    p,
                )
            end
        end
    end

    return v
end

In [ ]:
function is_independent(
    state::ModBasis,
    row,
)
    v = Int.(collect(row))
    reduce_with_basis!(v, state)
    return any(value -> value != 0, v)
end

In [ ]:
function add_row!(
    state::ModBasis,
    row,
)
    p = state.p
    ncols = state.ncols

    v = Int.(collect(row))
    reduce_with_basis!(v, state)

    pivot = findfirst(value -> value != 0, v)

    if pivot === nothing
        return false
    end

    pivot_inverse = invmod(v[pivot], p)

    @inbounds for j in pivot:ncols
        v[j] = mod(v[j] * pivot_inverse, p)
    end

    @inbounds for j in 1:ncols
        if j != pivot && state.basis[j] !== nothing
            basis_row = state.basis[j]::Vector{Int}
            coefficient = basis_row[pivot]

            if coefficient != 0
                for k in pivot:ncols
                    basis_row[k] = mod(
                        basis_row[k] - coefficient * v[k],
                        p,
                    )
                end
            end
        end
    end

    state.basis[pivot] = v
    state.rank += 1

    return true
end

In [ ]:
function rank_mod(
    A::AbstractMatrix{Int},
    rows::AbstractVector{<:Integer},
    p::Int,
)
    state = ModBasis(p, size(A, 2))

    for row in rows
        add_row!(state, view(A, row, :))
    end

    return state.rank
end

In [ ]:
function rank_mod_allrows(
    A::AbstractMatrix{Int},
    p::Int,
)
    return rank_mod(
        A,
        collect(axes(A, 1)),
        p,
    )
end

In [ ]:
# ============================================================
# Prepare modular copies and choose reliable primes.
# ============================================================

PRIME_CANDIDATES = [
    1_000_003,
    1_000_033,
    1_000_037,
    1_000_039,
    1_000_081,
    1_000_099,
    1_000_117,
    1_000_121,
    1_000_133,
    1_000_151,
]


function prepare_modular_data(
    N,
    T;
    prime_candidates = PRIME_CANDIDATES,
    number_of_primes::Int = 3,
    float_tolerance::Real = 1e-12,
)
    valid_primes = Int[]
    N_mod = Dict{Int, Matrix{Int}}()
    T_mod = Dict{Int, Matrix{Int}}()

    for p in prime_candidates
        try
            Np = matrix_mod_prime(
                N,
                p;
                float_tolerance = float_tolerance,
            )
            Tp = matrix_mod_prime(
                T,
                p;
                float_tolerance = float_tolerance,
            )

            N_mod[p] = Np
            T_mod[p] = Tp
            push!(valid_primes, p)

            length(valid_primes) >= number_of_primes && break
        catch error_object
            @warn(
                "Skipping prime $p.",
                exception = (
                    error_object,
                    catch_backtrace(),
                ),
            )
        end
    end

    length(valid_primes) >= 1 || error(
        "No usable prime was found. Add more prime candidates.",
    )

    return valid_primes, N_mod, T_mod
end

In [ ]:
valid_primes, N_mod, T_mod = prepare_modular_data(N, T)

full_rank_records = [
    (
        p = p,
        rank_N = rank_mod_allrows(N_mod[p], p),
        rank_T = rank_mod_allrows(T_mod[p], p),
    )
    for p in valid_primes
]

In [ ]:
for record in full_rank_records
    println(
        "p = ",
        record.p,
        ": rank(N) = ",
        record.rank_N,
        ", rank(T) = ",
        record.rank_T,
    )
end

In [ ]:
reference_rank_N = maximum(
    record.rank_N for record in full_rank_records
)
reference_rank_T = maximum(
    record.rank_T for record in full_rank_records
)

In [ ]:
verification_primes = [
    record.p
    for record in full_rank_records
    if (
        record.rank_N == reference_rank_N &&
        record.rank_T == reference_rank_T
    )
]

In [ ]:
isempty(verification_primes) && error(
    "The maximum modular ranks did not occur at the same prime. " *
    "Add more prime candidates.",
)

In [ ]:
search_prime = first(verification_primes)

In [ ]:
full_ranks = Dict(
    record.p => (
        rank_N = record.rank_N,
        rank_T = record.rank_T,
    )
    for record in full_rank_records
)

In [ ]:
println("Search prime = ", search_prime)
println("Verification primes = ", verification_primes)
println(
    "Reference full-rank gap = ",
    reference_rank_N - reference_rank_T,
)

In [ ]:
if reference_rank_N <= reference_rank_T
    @warn(
        "rank(N) is not larger than rank(T). The randomized search " *
        "will still run, but the greedy method is heuristic in this case " *
        "and may fail to find an existing positive-gap set.",
    )
end

In [ ]:
if length(verification_primes) == 1
    @warn(
        "Only one prime attained both reference ranks. " *
        "The search can run, but verification with multiple primes " *
        "is preferable.",
    )
end

In [ ]:
# ============================================================
# Gap and verification functions.
# ============================================================

function gap_from_ranks(
    rank_N_I::Int,
    rank_T_I::Int,
    rank_N_full::Int,
    rank_T_full::Int,
)
    return (
        (rank_N_full - rank_N_I) -
        (rank_T_full - rank_T_I)
    )
end

In [ ]:
function verify_set(
    N_mod,
    T_mod,
    S;
    primes,
    full_ranks,
)
    first_prime = first(primes)
    number_of_rows = size(N_mod[first_prime], 1)

    S_sorted = sort(unique(Int.(collect(S))))
    I = setdiff(
        collect(1:number_of_rows),
        S_sorted,
    )

    records = NamedTuple[]

    for p in primes
        rank_N_I = rank_mod(N_mod[p], I, p)
        rank_T_I = rank_mod(T_mod[p], I, p)

        rank_N_full = full_ranks[p].rank_N
        rank_T_full = full_ranks[p].rank_T

        drop_N = rank_N_full - rank_N_I
        drop_T = rank_T_full - rank_T_I

        push!(
            records,
            (
                p = p,
                rank_N_full = rank_N_full,
                rank_T_full = rank_T_full,
                rank_N_I = rank_N_I,
                rank_T_I = rank_T_I,
                drop_N = drop_N,
                drop_T = drop_T,
                gap = drop_N - drop_T,
            ),
        )
    end

    return I, S_sorted, records
end

In [ ]:
# ============================================================
# Randomized greedy search for a prescribed target gap.
#
# The algorithm tries to make I as large as possible while
# retaining a gap that matches the requested target condition.
# Consequently, S = [n] \ I is made as small as possible by
# this randomized greedy heuristic.
# ============================================================

function greedy_target_gap(
    Np::AbstractMatrix{Int},
    Tp::AbstractMatrix{Int};
    p::Int,
    rank_N_full::Int,
    rank_T_full::Int,
    target_gap::Int,
    rng = Random.default_rng(),
    maxS::Union{Nothing, Int} = nothing,
    make_S_exact_maxS::Bool = false,
)
    number_of_rows = size(Np, 1)

    @assert size(Tp, 1) == number_of_rows
    @assert target_gap >= 1 "target_gap must be positive."

    if maxS !== nothing
        @assert 0 <= maxS <= number_of_rows
    end

    if make_S_exact_maxS
        maxS === nothing && error(
            "make_S_exact_maxS=true requires an integer maxS.",
        )
    end

    state_N = ModBasis(p, size(Np, 2))
    state_T = ModBasis(p, size(Tp, 2))

    I = Int[]
    current_gap = rank_N_full - rank_T_full

    # Several passes are useful because a row rejected in an early
    # pass can become dependent, and hence acceptable, in a later pass.
    changed = true

    while changed
        changed = false
        remaining_rows = setdiff(
            collect(1:number_of_rows),
            I,
        )
        shuffle!(rng, remaining_rows)

        for row in remaining_rows
            independent_N = is_independent(
                state_N,
                view(Np, row, :),
            ) ? 1 : 0

            independent_T = is_independent(
                state_T,
                view(Tp, row, :),
            ) ? 1 : 0

            new_rank_N_I = state_N.rank + independent_N
            new_rank_T_I = state_T.rank + independent_T

            new_gap = gap_from_ranks(
                new_rank_N_I,
                new_rank_T_I,
                rank_N_full,
                rank_T_full,
            )

            accept_row = if current_gap >= target_gap
                new_gap >= target_gap
            else
                # Before reaching the target, retain only
                # nonworsening moves.
                new_gap >= current_gap
            end

            if accept_row
                add_row!(state_N, view(Np, row, :))
                add_row!(state_T, view(Tp, row, :))
                push!(I, row)
                current_gap = new_gap
                changed = true
            end
        end
    end

    sort!(I)
    S = setdiff(
        collect(1:number_of_rows),
        I,
    )

    # Optionally enlarge S to exactly maxS while preserving
    # gap >= target_gap.
    if (
        make_S_exact_maxS &&
        maxS !== nothing &&
        length(S) < maxS &&
        current_gap >= target_gap
    )
        candidate_rows = copy(I)
        shuffle!(rng, candidate_rows)

        for row in candidate_rows
            length(S) >= maxS && break

            S_candidate = sort!(vcat(S, row))
            I_candidate = setdiff(
                collect(1:number_of_rows),
                S_candidate,
            )

            rank_N_I = rank_mod(Np, I_candidate, p)
            rank_T_I = rank_mod(Tp, I_candidate, p)

            candidate_gap = gap_from_ranks(
                rank_N_I,
                rank_T_I,
                rank_N_full,
                rank_T_full,
            )

            if candidate_gap >= target_gap
                S = S_candidate
                I = I_candidate
                current_gap = candidate_gap
            end
        end
    end

    return sort(I), sort(S), current_gap
end

In [ ]:
# ============================================================
# Search for several sets having a prescribed target gap.
#
# gap_mode = :exact
#     requires verified gap == target_gap.
#
# gap_mode = :at_least
#     requires verified gap >= target_gap.
# ============================================================

function gap_matches(
    actual_gap::Int,
    target_gap::Int,
    gap_mode::Symbol,
)
    if gap_mode === :exact
        return actual_gap == target_gap
    elseif gap_mode === :at_least
        return actual_gap >= target_gap
    else
        error("gap_mode must be :exact or :at_least.")
    end
end

In [ ]:
function save_target_gap_results(
    results,
    outdir::AbstractString;
    target_gap::Int,
    gap_mode::Symbol,
    number_of_rows::Int,
    verification_primes,
)
    isempty(results) && return nothing

    mkpath(outdir)

    summary_lines = String[
        join(
            [
                "id",
                "target_gap",
                "gap_mode",
                "number_of_rows",
                "size_S",
                "size_I",
                "rank_N_full",
                "rank_T_full",
                "rank_N_I",
                "rank_T_I",
                "drop_N",
                "drop_T",
                "gap",
                "verification_primes",
            ],
            ",",
        ),
    ]

    for result in results
        id = result.id
        I = result.I
        S = result.S
        record0 = first(result.verification)
        set_id = lpad(id, 2, '0')

        writedlm(
            joinpath(
                outdir,
                "gap_$(record0.gap)_set_$(set_id)_I_1_based.txt",
            ),
            I,
        )

        writedlm(
            joinpath(
                outdir,
                "gap_$(record0.gap)_set_$(set_id)_S_1_based.txt",
            ),
            S,
        )

        push!(
            summary_lines,
            join(
                [
                    string(id),
                    string(target_gap),
                    string(gap_mode),
                    string(number_of_rows),
                    string(length(S)),
                    string(length(I)),
                    string(record0.rank_N_full),
                    string(record0.rank_T_full),
                    string(record0.rank_N_I),
                    string(record0.rank_T_I),
                    string(record0.drop_N),
                    string(record0.drop_T),
                    string(record0.gap),
                    "\"" * join(verification_primes, ";") * "\"",
                ],
                ",",
            ),
        )
    end

    open(
        joinpath(outdir, "target_gap_summary.csv"),
        "w",
    ) do io
        for line in summary_lines
            println(io, line)
        end
    end

    return nothing
end

In [ ]:
function _search_many_target_gap_core(
    N_mod,
    T_mod;
    search_prime::Int,
    verification_primes,
    full_ranks,
    target_gap::Int,
    gap_mode::Symbol = :exact,
    nsets::Int = 10,
    maxS::Union{Nothing, Int} = nothing,
    trials::Int = 1000,
    seed::Int = 1234,
    make_S_exact_maxS::Bool = false,
    outdir::AbstractString = "target_gap_outputs",
    save_outputs::Bool = true,
)
    @assert target_gap >= 1
    @assert nsets >= 1
    @assert trials >= 1

    gap_mode in (:exact, :at_least) || error(
        "gap_mode must be :exact or :at_least.",
    )

    rng = MersenneTwister(seed)

    Np = N_mod[search_prime]
    Tp = T_mod[search_prime]
    number_of_rows = size(Np, 1)

    if maxS !== nothing
        @assert 0 <= maxS <= number_of_rows
    end

    seen = Set{String}()
    results = NamedTuple[]

    for trial in 1:trials
        I, S, search_gap = greedy_target_gap(
            Np,
            Tp;
            p = search_prime,
            rank_N_full =
                full_ranks[search_prime].rank_N,
            rank_T_full =
                full_ranks[search_prime].rank_T,
            target_gap = target_gap,
            rng = rng,
            maxS = maxS,
            make_S_exact_maxS =
                make_S_exact_maxS,
        )

        gap_matches(
            search_gap,
            target_gap,
            gap_mode,
        ) || continue

        if maxS !== nothing && length(S) > maxS
            continue
        end

        I_verified, S_verified, records = verify_set(
            N_mod,
            T_mod,
            S;
            primes = verification_primes,
            full_ranks = full_ranks,
        )

        @assert I_verified == I
        @assert S_verified == S

        record0 = first(records)

        # Require the same submatrix ranks and gap for all
        # verification primes.
        ranks_are_consistent = all(
            record ->
                record.rank_N_full == record0.rank_N_full &&
                record.rank_T_full == record0.rank_T_full &&
                record.rank_N_I == record0.rank_N_I &&
                record.rank_T_I == record0.rank_T_I &&
                record.gap == record0.gap,
            records,
        )

        ranks_are_consistent || continue

        gap_matches(
            record0.gap,
            target_gap,
            gap_mode,
        ) || continue

        key = join(S, ",")
        key in seen && continue
        push!(seen, key)

        id = length(results) + 1

        push!(
            results,
            (
                id = id,
                target_gap = target_gap,
                actual_gap = record0.gap,
                I = I,
                S = S,
                verification = records,
            ),
        )

        println(
            "Found set ",
            id,
            " for target gap ",
            target_gap,
            " in trial ",
            trial,
            ": |S| = ",
            length(S),
            ", |I| = ",
            length(I),
            ", verified gap = ",
            record0.gap,
        )

        length(results) >= nsets && break
    end

    if save_outputs && !isempty(results)
        save_target_gap_results(
            results,
            outdir;
            target_gap = target_gap,
            gap_mode = gap_mode,
            number_of_rows = number_of_rows,
            verification_primes = verification_primes,
        )

        println(
            "Saved target-gap outputs in: ",
            abspath(outdir),
        )
    end

    return results
end

In [ ]:
# ============================================================
# Public interfaces.
# ============================================================

function search_many_target_gap(
    N_mod,
    T_mod;
    target_gap::Int,
    gap_mode::Symbol = :exact,
    nsets::Int = 10,
    maxS::Union{Nothing, Int} = nothing,
    trials::Int = 1000,
    seed::Int = 1234,
    make_S_exact_maxS::Bool = false,
    outdir::AbstractString = "target_gap_outputs",
)
    return _search_many_target_gap_core(
        N_mod,
        T_mod;
        search_prime = search_prime,
        verification_primes = verification_primes,
        full_ranks = full_ranks,
        target_gap = target_gap,
        gap_mode = gap_mode,
        nsets = nsets,
        maxS = maxS,
        trials = trials,
        seed = seed,
        make_S_exact_maxS = make_S_exact_maxS,
        outdir = outdir,
        save_outputs = true,
    )
end

In [ ]:
# Backward-compatible interface: positive gap means target_gap = 1
# with the condition gap >= 1.
function search_many_positive_gap(
    N_mod,
    T_mod;
    nsets::Int = 10,
    maxS::Union{Nothing, Int} = nothing,
    trials::Int = 1000,
    seed::Int = 1234,
    make_S_exact_maxS::Bool = false,
    outdir::AbstractString = "positive_gap_outputs",
)
    return search_many_target_gap(
        N_mod,
        T_mod;
        target_gap = 1,
        gap_mode = :at_least,
        nsets = nsets,
        maxS = maxS,
        trials = trials,
        seed = seed,
        make_S_exact_maxS = make_S_exact_maxS,
        outdir = outdir,
    )
end

In [ ]:
function modular_column_space_inclusion(
    N_mod,
    T_mod;
    primes,
    full_ranks,
)
    records = NamedTuple[]

    for p in primes
        rank_N = full_ranks[p].rank_N
        rank_NT = rank_mod_allrows(
            hcat(N_mod[p], T_mod[p]),
            p,
        )

        push!(
            records,
            (
                p = p,
                rank_N = rank_N,
                rank_NT = rank_NT,
                included = rank_NT == rank_N,
            ),
        )
    end

    return all(record -> record.included, records), records
end

In [ ]:
function save_largest_gap_attempts(
    attempts,
    outdir::AbstractString,
)
    mkpath(outdir)

    open(
        joinpath(outdir, "largest_gap_attempts.csv"),
        "w",
    ) do io
        println(
            io,
            "target_gap,found_count,best_actual_gap,min_size_S,max_size_I",
        )

        for attempt in attempts
            println(
                io,
                join(
                    [
                        string(attempt.target_gap),
                        string(attempt.found_count),
                        string(attempt.best_actual_gap),
                        string(attempt.min_size_S),
                        string(attempt.max_size_I),
                    ],
                    ",",
                ),
            )
        end
    end

    return nothing
end

In [ ]:
function search_largest_gap(
    N_mod,
    T_mod;
    gap_min::Int = 1,
    gap_max::Int = 50,
    maxS::Union{Nothing, Int} = nothing,
    nsets::Int = 10,
    trials_per_gap::Int = 1000,
    seed::Int = 1234,
    make_S_exact_maxS::Bool = false,
    outdir::AbstractString = "largest_gap_outputs",
)
    @assert 1 <= gap_min <= gap_max
    @assert nsets >= 1
    @assert trials_per_gap >= 1

    number_of_rows = size(
        N_mod[search_prime],
        1,
    )

    if maxS !== nothing
        @assert 0 <= maxS <= number_of_rows
    end

    inclusion_verified, inclusion_records =
        modular_column_space_inclusion(
            N_mod,
            T_mod;
            primes = verification_primes,
            full_ranks = full_ranks,
        )

    full_gap = (
        full_ranks[search_prime].rank_N -
        full_ranks[search_prime].rank_T
    )

    effective_gap_max = gap_max

    if inclusion_verified
        effective_gap_max = min(
            gap_max,
            full_gap,
        )

        println(
            "Verified modulo all selected primes that ",
            "col(T) is contained in col(N).",
        )
        println(
            "Therefore the gap cannot exceed ",
            full_gap,
            ".",
        )
    else
        @warn(
            "col(T) ⊆ col(N) was not verified for every selected prime. " *
            "The search will still use the requested gap range.",
        )
    end

    attempts = NamedTuple[]

    if effective_gap_max < gap_min
        @warn(
            "The effective upper gap bound is smaller than gap_min.",
        )

        save_largest_gap_attempts(
            attempts,
            outdir,
        )

        return (
            best_gap = nothing,
            found = NamedTuple[],
            attempts = attempts,
            inclusion_verified = inclusion_verified,
            inclusion_records = inclusion_records,
            full_gap = full_gap,
        )
    end

    println(
        "Searching target gaps from ",
        effective_gap_max,
        " down to ",
        gap_min,
        " under |S| <= ",
        maxS === nothing ? "no bound" : string(maxS),
        ".",
    )

    for target_gap in effective_gap_max:-1:gap_min
        println()
        println(
            "Searching exact gap = ",
            target_gap,
            "...",
        )

        target_outdir = joinpath(
            outdir,
            "gap_$(target_gap)",
        )

        results = _search_many_target_gap_core(
            N_mod,
            T_mod;
            search_prime = search_prime,
            verification_primes = verification_primes,
            full_ranks = full_ranks,
            target_gap = target_gap,
            gap_mode = :exact,
            nsets = nsets,
            maxS = maxS,
            trials = trials_per_gap,
            seed = seed + target_gap,
            make_S_exact_maxS =
                make_S_exact_maxS,
            outdir = target_outdir,
            save_outputs = true,
        )

        if isempty(results)
            push!(
                attempts,
                (
                    target_gap = target_gap,
                    found_count = 0,
                    best_actual_gap = missing,
                    min_size_S = missing,
                    max_size_I = missing,
                ),
            )
        else
            push!(
                attempts,
                (
                    target_gap = target_gap,
                    found_count = length(results),
                    best_actual_gap = maximum(
                        result.actual_gap
                        for result in results
                    ),
                    min_size_S = minimum(
                        length(result.S)
                        for result in results
                    ),
                    max_size_I = maximum(
                        length(result.I)
                        for result in results
                    ),
                ),
            )

            save_largest_gap_attempts(
                attempts,
                outdir,
            )

            println()
            println(
                "Largest exact gap found in the requested range: ",
                target_gap,
            )

            return (
                best_gap = target_gap,
                found = results,
                attempts = attempts,
                inclusion_verified = inclusion_verified,
                inclusion_records = inclusion_records,
                full_gap = full_gap,
            )
        end
    end

    save_largest_gap_attempts(
        attempts,
        outdir,
    )

    @warn(
        "No exact gap in the requested range was found under the " *
        "current maxS and trial limits. This is a heuristic search, " *
        "so failure is not a proof of nonexistence.",
    )

    return (
        best_gap = nothing,
        found = NamedTuple[],
        attempts = attempts,
        inclusion_verified = inclusion_verified,
        inclusion_records = inclusion_records,
        full_gap = full_gap,
    )
end

### Search for the index set, its complement and the corresponding rank gap.

After running the preceding cells, the following command searches for an index set $I\subseteq[n]$, its complement
$$S=[n]\setminus I,$$
and the corresponding rank gap.  Thus, $I\cup S=[n].$

For the matrices $N$ and $T$, the rank gap associated with $I$ is
$$
\operatorname{gap}(I)
=\left(\operatorname{rank}(N)-\operatorname{rank}(N[I,:])\right)
-
\left(\operatorname{rank}(T)-\operatorname{rank}(T[I,:])\right).
$$

The final command contains the following main parameters:

- `gap_min`, `gap_max`: the prescribed range of exact gaps;
- `maxS`: the upper bound on $|S|$;
- `nsets`: the target number of distinct index sets to be retained for
  the first successful gap;
- `trials_per_gap`: the maximum number of randomized trials performed
  for each candidate gap;
- `seed`: the random seed used to make the search reproducible;
- `make_S_exact_maxS`: if set to `true`, the program attempts to enlarge
  $S$ until $|S|=\texttt{maxS}$, while preserving the target gap;
- `outdir`: the root directory in which the search results are saved.

The program searches the exact gaps in descending order and stops as soon
as at least one verified set is found. Therefore, it returns the largest
exact gap discovered in the prescribed range subject to the bound on $|S|$.

This is a randomized heuristic. Failure to find a particular gap within
the prescribed number of trials does not prove that such a gap does not
exist.

In [ ]:
# ============================================================
# Search settings
# ============================================================
# Search for I, S and gap(I)  
# subject to |S| <= maxS and gap_min<= gap(I) <= gap_max

largest_gap_result = search_largest_gap(
    N_mod,
    T_mod;
    gap_min = 1,
    gap_max = 2,
    maxS = 300,
    nsets = 20,
    trials_per_gap = 1000,
    seed = 20260704,
    make_S_exact_maxS = false,
    outdir = "gap1to2_le300_date"
)